# 11. Final Comparison and Qualitative LLM Evaluation
This notebook represents the final phase of the academic production analysis project.

**Objectives:**
1. **Consolidate Quantitative Metrics:** Extract and compare all metrics generated by the different methods (both geometric and semantic) in the `outputs/` folder.
2. **Qualitative Evaluation (LLM):** Use DeepSeek to analyze the top-terms of all models that generate topics. The LLM will not only assign a label to the cluster, but will also evaluate the **cohesion** (well-defined niche vs. diffuse group) of the words to determine the true human interpretability of each model.

## 11.1 Environment Setup

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

def check_and_install(package_name, pip_name=None):
    if pip_name is None:
        pip_name = package_name
    if importlib.util.find_spec(package_name) is None:
        try:
            from google.colab import drive
            print(f"Installing {pip_name} in Colab...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        except ImportError:
            print(f"Warning: '{package_name}' is missing. Make sure to install '{pip_name}' in your local .venv")

check_and_install("tenacity")
check_and_install("openai")
check_and_install("seaborn")
check_and_install("matplotlib")

BASE_PATH = Path("..") # Assuming execution from /notebooks
if str(BASE_PATH) not in sys.path:
    sys.path.append(str(BASE_PATH))

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Visualization configurations
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

from utils.llm_labeler import TopTermsTopicLabeler

## 11.2 Consolidation of Quantitative Metrics
We scan the `outputs/` directory to collect the metrics from all models (Classical, BERTopic, FASTopic, etc.).

In [ ]:
OUTPUTS_DIR = BASE_PATH / "outputs"
metrics_files = list(OUTPUTS_DIR.glob("*_metrics.csv"))

print(f"Metrics files found: {len(metrics_files)}")

all_metrics = []
for file_path in metrics_files:
    filename = file_path.name
    try:
        df = pd.read_csv(file_path)
        # Identify the source based on the filename
        pipeline_stage = filename.split('_')[0] # e.g. 04, 05, 07
        
        # Iterate through the models within the file
        for _, row in df.iterrows():
            model_name = row.get('model', row.get('run_id', 'Unknown'))
            
            # Collect available metrics
            metrics_dict = {
                'notebook': pipeline_stage,
                'file_source': filename,
                'model_name': model_name,
                'silhouette': row.get('silhouette_score', np.nan),
                'calinski_harabasz': row.get('calinski_harabasz_score', np.nan),
                'davies_bouldin': row.get('davies_bouldin_score', np.nan),
                'npmi': row.get('NPMI', row.get('npmi', np.nan)),
                'topic_diversity': row.get('Topic_Diversity', row.get('topic_diversity', np.nan)),
            }
            all_metrics.append(metrics_dict)
            
    except Exception as e:
        print(f"Error reading {filename}: {e}")

metrics_df = pd.DataFrame(all_metrics)

print("--- Consolidated Metrics Summary ---")
display(metrics_df)

### QUANTITATIVE METRICS INTERPRETATION (To be completed by the user)
*(Write your observations here about which models stand out in Silhouette vs NPMI)*

## 11.3 Visual Analysis of Models

In [ ]:
# Plot 1: Comparison of Geometric Metrics (Silhouette)
# Useful for classical vs modern clustering methods
geom_df = metrics_df.dropna(subset=['silhouette'])
if not geom_df.empty:
    plt.figure(figsize=(10, 6))
    sns.barplot(data=geom_df, x='silhouette', y='model_name', hue='notebook', dodge=False, palette='viridis')
    plt.title("Clustering Quality Comparison (Silhouette Score)\nHigher is better")
    plt.xlabel("Silhouette Score")
    plt.ylabel("Model")
    plt.tight_layout()
    plt.show()
else:
    print("No Silhouette metrics available to plot.")

In [ ]:
# Plot 2: Comparison of Topic Coherence (NPMI)
# Useful to compare the internal interpretability of topic generators
topic_df = metrics_df.dropna(subset=['npmi'])
if not topic_df.empty:
    plt.figure(figsize=(10, 6))
    sns.barplot(data=topic_df, x='npmi', y='model_name', hue='notebook', dodge=False, palette='magma')
    plt.title("Semantic Coherence Comparison (NPMI)\nHigher is better")
    plt.xlabel("Normalized Pointwise Mutual Information (NPMI)")
    plt.ylabel("Model")
    plt.tight_layout()
    plt.show()
else:
    print("No consolidated NPMI metrics available to plot together.")

## 11.4 Qualitative Evaluation with LLM (Labeling and Cohesion)
We load the `_top_terms.csv` files generated by all models to cross-evaluate them using DeepSeek.

In [ ]:
top_terms_files = list(OUTPUTS_DIR.glob("*_top_terms.csv"))
print(f"Top terms files found: {len(top_terms_files)}")

# Consolidate all top terms into a dictionary
models_top_terms = {}

for file_path in top_terms_files:
    filename = file_path.name
    # Extract the clean model name from the filename: e.g. 04_classical_lda_top_terms.csv -> classical_lda
    model_key = filename.replace("_top_terms.csv", "")
    
    try:
        df_terms = pd.read_csv(file_path)
        # Normalize the cluster column name
        cluster_col = 'cluster_id' if 'cluster_id' in df_terms.columns else 'cluster'
        
        if cluster_col in df_terms.columns and 'top_terms' in df_terms.columns:
            models_top_terms[model_key] = {
                'df': df_terms,
                'cluster_col': cluster_col,
                'terms_col': 'top_terms'
            }
        else:
            print(f"Warning: Unexpected columns in {filename}")
    except Exception as e:
        print(f"Error processing {filename}: {e}")

In [ ]:
# Run the LLM Labeler asynchronously for all models
labeler = TopTermsTopicLabeler(
    semaphore_limit=15, 
    checkpoint_dir=str(BASE_PATH / "data/processed"), 
    checkpoint_filename="11_llm_top_terms_evaluation.json"
)

# Dictionary to store the consolidated results of all models
all_llm_evaluations = {}

for model_key, data in models_top_terms.items():
    print(f"\n--- Evaluating model: {model_key} ---")
    df_terms = data['df']
    cluster_col = data['cluster_col']
    terms_col = data['terms_col']
    
    # Some models use -1 for noise (HDBSCAN), we ignore noise in the evaluation
    evaluations = await labeler.generate_labels(
        df=df_terms, 
        id_col=cluster_col, 
        terms_col=terms_col, 
        noise_id=-1
    )
    all_llm_evaluations[model_key] = evaluations

## 11.5 Consolidation and Export of LLM Evaluations
We join the LLM results along with the original top terms to facilitate manual reading and comparison.

In [ ]:
final_evaluations_list = []

for model_key, data in models_top_terms.items():
    df_terms = data['df']
    cluster_col = data['cluster_col']
    terms_col = data['terms_col']
    
    evals = all_llm_evaluations.get(model_key, {})
    
    for _, row in df_terms.iterrows():
        c_id = row[cluster_col]
        
        # Ignore noise
        if c_id == -1:
            continue
            
        c_str = str(c_id)
        
        if c_str in evals:
            res = evals[c_str]
            final_evaluations_list.append({
                'model_source': model_key,
                'topic_id': c_id,
                'top_terms': row[terms_col],
                'llm_label': res.get('label', ''),
                'cohesion_score': res.get('cohesion_score', ''),
                'reasoning': res.get('reasoning', '')
            })

llm_eval_df = pd.DataFrame(final_evaluations_list)

# Export to CSV for easy visualization
OUTPUT_EVAL_CSV = OUTPUTS_DIR / "11_final_llm_evaluations.csv"
llm_eval_df.to_csv(OUTPUT_EVAL_CSV, index=False)
print(f"Evaluations successfully exported to: {OUTPUT_EVAL_CSV}")

### Cohesion Comparison by Model
We can analyze what percentage of topics were rated as having "High" vs "Low" cohesion by each model.

In [ ]:
if not llm_eval_df.empty:
    cohesion_counts = llm_eval_df.groupby(['model_source', 'cohesion_score']).size().unstack(fill_value=0)
    display(cohesion_counts)
    
    # Stacked bar chart of cohesion
    cohesion_counts.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='Set2')
    plt.title("Proportion of Semantic Cohesion Evaluated by LLM per Model")
    plt.xlabel("Model")
    plt.ylabel("Number of Topics")
    plt.legend(title="Cohesion Score")
    plt.tight_layout()
    plt.show()

### FINAL INTERPRETATION (To be completed by the user)
*(Write your conclusions here about which is the definitive SOTA model for this use case based on the quantitative and qualitative LLM evaluation)*